# 19 — A2A hands-on with the official Python SDK

The official SDK (`a2a-sdk`, v1.x) implements the v1.0 protocol: servers plug an `AgentExecutor` into request handlers that own the task lifecycle, and clients come from a `create_client` factory that resolves the Agent Card for you. As with notebook 17, the live cell needs **no API key** — the server is a local subprocess speaking JSON-RPC over `127.0.0.1`.

Install the optional extra first: `pip install -e '.[a2a]'`.

Concept mapping to notebook 18: `AgentCard`/`create_agent_card_routes` ≙ `ToyAgentCard` + well-known discovery, `TaskUpdater.update_status/add_artifact` ≙ the toy server's state transitions and artifacts, `create_client(...).send_message` ≙ `ToyA2AClient.send_message` — same wire concepts, real transport, typed (protobuf-generated) messages.

Primary sources: [a2a-python SDK](https://github.com/a2aproject/a2a-python) and the [helloworld sample](https://github.com/a2aproject/a2a-samples/tree/main/samples/python/agents/helloworld).

In [ ]:
import sys
from importlib.util import find_spec
from pathlib import Path
sys.path.insert(0, str(Path.cwd() / 'notebooks'))

A2A_AVAILABLE = find_spec('a2a') is not None
if not A2A_AVAILABLE:
    print("Install the optional extra to run the live cell: pip install -e '.[a2a]'")

server_code = '''
import uvicorn
from starlette.applications import Starlette

from a2a.helpers import get_message_text, new_task_from_user_message, new_text_part
from a2a.server.agent_execution import AgentExecutor, RequestContext
from a2a.server.events import EventQueue
from a2a.server.request_handlers import DefaultRequestHandler
from a2a.server.routes import create_agent_card_routes, create_jsonrpc_routes
from a2a.server.tasks import InMemoryTaskStore, TaskUpdater
from a2a.types import AgentCapabilities, AgentCard, AgentInterface, AgentSkill, TaskState


class ForecastExecutor(AgentExecutor):
    async def execute(self, context: RequestContext, event_queue: EventQueue) -> None:
        task = context.current_task or new_task_from_user_message(context.message)
        if not context.current_task:
            await event_queue.enqueue_event(task)
        updater = TaskUpdater(event_queue=event_queue, task_id=task.id, context_id=task.context_id)
        await updater.update_status(state=TaskState.TASK_STATE_WORKING)
        city = get_message_text(context.message).lower().strip()
        forecast = {'oslo': 'Snow, -3C', 'lisbon': 'Sunny, 21C'}.get(city, 'unknown city')
        await updater.add_artifact(parts=[new_text_part(text=forecast, media_type='text/plain')])
        await updater.update_status(state=TaskState.TASK_STATE_COMPLETED)

    async def cancel(self, context: RequestContext, event_queue: EventQueue) -> None:
        raise NotImplementedError('cancel is not supported')


card = AgentCard(
    name='weather-lab-a2a',
    description='Deterministic forecast agent for the lab.',
    version='0.1.0',
    default_input_modes=['text/plain'],
    default_output_modes=['text/plain'],
    capabilities=AgentCapabilities(streaming=False),
    supported_interfaces=[
        AgentInterface(protocol_binding='JSONRPC', url='http://127.0.0.1:9999', protocol_version='1.0')
    ],
    skills=[
        AgentSkill(
            id='get_forecast',
            name='Get forecast',
            description='Forecast for a known city.',
            tags=['weather'],
        )
    ],
)

handler = DefaultRequestHandler(
    agent_executor=ForecastExecutor(), task_store=InMemoryTaskStore(), agent_card=card
)
routes = [*create_agent_card_routes(card), *create_jsonrpc_routes(handler, '/')]

if __name__ == '__main__':
    uvicorn.run(Starlette(routes=routes), host='127.0.0.1', port=9999, log_level='error')
'''
print(server_code)

In [ ]:
# Live: spawn the server as a subprocess, discover its card, delegate a task.
if A2A_AVAILABLE:
    import asyncio, subprocess, tempfile
    import httpx

    server_path = Path(tempfile.mkdtemp()) / 'forecast_a2a_server.py'
    server_path.write_text(server_code)
    proc = subprocess.Popen([sys.executable, str(server_path)])
    try:
        async def wait_ready():
            async with httpx.AsyncClient() as http:
                for _ in range(40):
                    try:
                        response = await http.get('http://127.0.0.1:9999/.well-known/agent-card.json')
                        if response.status_code == 200:
                            return response.json()
                    except httpx.TransportError:
                        pass
                    await asyncio.sleep(0.25)
                raise RuntimeError('server did not start')

        card = await wait_ready()
        print('discovered:', card['name'], '| binding:', card['supportedInterfaces'][0]['protocolBinding'])

        from a2a.client import ClientConfig, create_client
        from a2a.helpers import get_artifact_text, new_text_message
        from a2a.types import Role, SendMessageRequest, TaskState

        client = await create_client('http://127.0.0.1:9999', ClientConfig(streaming=False))
        request = SendMessageRequest(message=new_text_message('Oslo', role=Role.ROLE_USER))
        async for response in client.send_message(request):
            if response.WhichOneof('payload') == 'task':
                task = response.task
                print('task state:', TaskState.Name(task.status.state))
                for artifact in task.artifacts:
                    print('artifact:', get_artifact_text(artifact))
        await client.close()
    finally:
        proc.terminate()
else:
    print('skipped: a2a-sdk not installed')

**Design translation.** The exchange above is notebook 18's toy session with real framing: the card came from the well-known URI, `SendMessage` returned a task that moved `SUBMITTED → WORKING → COMPLETED`, and the forecast arrived as an artifact part. What A2A does *not* solve remains your architecture: which peer to trust with which task (04), how to verify what comes back (03), and when a human must approve (05). Track B frameworks ship A2A integrations, so an executor you write once can front agents built with any of them.